# ADBC partition fetches — Arrow straight from the driver

ADBC (Arrow Database Connectivity) drivers return Arrow data directly from the database wire protocol, skipping the Python row objects a DBAPI driver produces. boti-data uses them for one thing: **partition fetches**, the hot path of every partitioned SQL load.

ADBC sits outside SQLAlchemy, which is where boti-data's read-only guards live. So the ADBC path is reachable only when the guards already hold, and it re-applies them itself:

| Guard | How ADBC honours it |
|---|---|
| `query_only=True` | required; the DSN must pass the same read-only validation (SQLite: `mode=ro`) |
| session read-only statement | the ADBC session gets the statement the SQLAlchemy engine issues on connect (`PRAGMA query_only`, `SET default_transaction_read_only`) |
| only planner-compiled `Select` partitions | raw SQL text never goes through ADBC |
| one connection per fetch | same rule as the workers' `NullPool` engines, so the fetch gate bounds open connections |

| # | Topic |
|---|---|
| 1 | Seed a database and build the read-only DSN |
| 2 | Eligibility: when ADBC is used and when it is not |
| 3 | The ADBC session refuses writes |
| 4 | Same partition, both paths: identical frames |
| 5 | Timing: ADBC vs the SQLAlchemy Arrow path |
| 6 | End to end through `SqlPartitionedLoader` |
| 7 | Driver errors fall back; `adbc=False` opts out |
| 8 | PostgreSQL (runs only if a local server accepts you) |

Install the drivers with `pip install "boti-data[adbc]"`.

In [1]:
import sys
import sqlite3
import tempfile
from pathlib import Path
from time import perf_counter

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
from sqlalchemy import select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

from boti_data.db import SqlDatabaseConfig, SqlPartitionedLoader, adbc_fetch
from boti_data.db import partitioned_execution
from boti_data.db.partitioned_execution import SqlPartitionExecutor
from boti_data.db.partitioned_types import SqlPartitionSpec
from boti_data.db.sql_config import WorkerSqlConfig

print("ADBC SQLite driver installed:", adbc_fetch.driver_installed("sqlite"))

ADBC SQLite driver installed: True


## 1. Seed a database and build the read-only DSN

`query_only=True` on SQLite requires the file-URI DSN with `mode=ro`. The ADBC driver opens the very same URI, so the file is read-only on both paths.

In [2]:
ROWS = 300_000
db_file = Path(tempfile.mkdtemp()) / "events.sqlite"
con = sqlite3.connect(db_file)
con.execute("create table events (id integer primary key, group_id integer, amount real, label text)")
con.executemany(
    "insert into events values (?, ?, ?, ?)",
    ((i, i % 97, i * 0.25, f"event-{i}") for i in range(1, ROWS + 1)),
)
con.commit()
con.close()

RO_DSN = f"sqlite:///file:{db_file}?mode=ro&uri=true"
META = {"id": "Int64", "group_id": "Int64", "amount": "Float64", "label": "string"}
PARTITION = SqlPartitionSpec(sql="select id, group_id, amount, label from events where id >= ? order by id", params=(1,))
print(RO_DSN)

sqlite:///file:/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmp78k9f3z_/events.sqlite?mode=ro&uri=true


## 2. Eligibility

`adbc_eligible` is the single decision point. All three must hold: the worker is read-only, the backend has an installed ADBC driver, and `adbc` was not set to `False`.

In [3]:
cases = {
    "query_only=True (default)": WorkerSqlConfig(connection_url=RO_DSN, query_only=True),
    "query_only=False": WorkerSqlConfig(connection_url=RO_DSN, query_only=False),
    "adbc=False": WorkerSqlConfig(connection_url=RO_DSN, query_only=True, adbc=False),
    "mysql (no ADBC driver)": WorkerSqlConfig(connection_url="mysql+pymysql://u:p@db/app", query_only=True),
}
pd.DataFrame(
    {"path": ["ADBC" if adbc_fetch.adbc_eligible(c) else "SQLAlchemy" for c in cases.values()]},
    index=list(cases),
)

,path
query_only=True (default),ADBC
query_only=False,SQLAlchemy
adbc=False,SQLAlchemy
mysql (no ADBC driver),SQLAlchemy


## 3. The ADBC session refuses writes

`open_read_only` connects, issues the backend's read-only statement, and closes on exit. A write through it fails at the database, not in Python.

In [4]:
config = WorkerSqlConfig(connection_url=RO_DSN, query_only=True)
with adbc_fetch.open_read_only(config) as connection, connection.cursor() as cursor:
    try:
        cursor.execute("insert into events values (0, 0, 0.0, 'x')")
    except Exception as exc:
        print(f"refused -> {type(exc).__name__}: {str(exc).splitlines()[0]}")

refused -> OperationalError: IO: Failed to step query: attempt to write a readonly database


## 4. Same partition, both paths

`fetch_partition` is the executor every partitioned load calls. With `adbc=False` it takes the SQLAlchemy Arrow path; otherwise ADBC. The frames must be identical.

In [5]:
def fetch(*, adbc: bool) -> pd.DataFrame:
    return SqlPartitionExecutor.fetch_partition(
        config=WorkerSqlConfig(connection_url=RO_DSN, query_only=True, adbc=adbc),
        gate_key="notebook",
        max_concurrent_fetches=1,
        partition=PARTITION,
        meta_dtypes=META,
        use_arrow=True,
    )

via_sqlalchemy = fetch(adbc=False)
via_adbc = fetch(adbc=True)
assert via_adbc.equals(via_sqlalchemy)
print("identical:", via_adbc.equals(via_sqlalchemy), "| dtypes:", dict(via_adbc.dtypes.astype(str)))
via_adbc.head()

identical: True | dtypes: {'id': 'Int64', 'group_id': 'Int64', 'amount': 'Float64', 'label': 'string'}


,id,group_id,amount,label
0,1,1,0.25,event-1
1,2,2,0.5,event-2
2,3,3,0.75,event-3
3,4,4,1.0,event-4
4,5,5,1.25,event-5


## 5. Timing

The SQLAlchemy Arrow path already builds Arrow in one columnar pass from DBAPI rows; ADBC skips the rows altogether.

In [6]:
def best_of_three(adbc: bool) -> float:
    times = []
    for _ in range(3):
        started = perf_counter()
        fetch(adbc=adbc)
        times.append(perf_counter() - started)
    return min(times)

timings = {"sqlalchemy-arrow": best_of_three(False), "adbc": best_of_three(True)}
print(f"{ROWS:,} rows  sqlalchemy-arrow: {timings['sqlalchemy-arrow']:.3f}s   adbc: {timings['adbc']:.3f}s   ({timings['sqlalchemy-arrow'] / timings['adbc']:.1f}x)")


300,000 rows  sqlalchemy-arrow: 0.197s   adbc: 0.078s   (2.5x)


## 6. End to end through `SqlPartitionedLoader`

Nothing changes in how a load is written. With `chunk_size` small enough to produce several partitions, each one goes through ADBC; we count the calls to prove it.

In [7]:
class Base(DeclarativeBase):
    pass

class Event(Base):
    __tablename__ = "events"
    id: Mapped[int] = mapped_column(primary_key=True)
    group_id: Mapped[int]
    amount: Mapped[float]
    label: Mapped[str]

adbc_calls = []
_real = partitioned_execution.fetch_partition_table
partitioned_execution.fetch_partition_table = lambda c, p, m: (adbc_calls.append(p.sql), _real(c, p, m))[1]
try:
    loader_config = SqlDatabaseConfig(connection_url=RO_DSN, query_only=True, poolclass="sqlalchemy.pool.NullPool")
    with SqlPartitionedLoader(loader_config) as loader:
        frame = loader.load(statement=select(Event).where(Event.group_id == 7), model=Event, chunk_size=1_000)
        result = frame.compute()
finally:
    partitioned_execution.fetch_partition_table = _real
print(f"rows={len(result)}  partitions={frame.npartitions}  ADBC fetches={len(adbc_calls)}")

[2026-11-09 16:34:56][WARNING][WorkerSqlConfig] worker_connection_env_var is not set; falling back to raw DSN. Credentials may be visible in scheduler logs.
rows=3093  partitions=4  ADBC fetches=4


## 7. Driver errors fall back; `adbc=False` opts out

A driver error on a partition logs a warning and that partition is fetched through SQLAlchemy, so a broken driver install degrades to the old speed rather than failing the load.

In [8]:
import adbc_driver_manager

def _boom(config, partition, meta):
    raise adbc_driver_manager.Error("driver exploded", status_code=adbc_driver_manager.AdbcStatusCode.INTERNAL)

partitioned_execution.fetch_partition_table = _boom
try:
    fallback = fetch(adbc=True)   # warning logged, SQLAlchemy path used
finally:
    partitioned_execution.fetch_partition_table = _real
print("rows via fallback:", len(fallback), "| opted out:", len(fetch(adbc=False)))

ADBC fetch failed, using SQLAlchemy for this partition: driver exploded


rows via fallback: 300000 | opted out: 300000


## 8. PostgreSQL

The planner compiles partitions for the SQLAlchemy driver (psycopg's `%(name)s`); ADBC PostgreSQL wants `$1..$n`. `positional` rewrites them. This cell runs only if a local server on 5432 accepts your user.

In [9]:
import getpass, socket

sql, params = adbc_fetch.positional(
    "select * from t where a = %(a_1)s and name like '%%x%%' and b in (%(b_1)s, %(b_2)s)",
    {"a_1": 1, "b_1": "p", "b_2": "q"},
    "postgresql",
)
print(sql, params)

def _local_pg():
    try:
        with socket.create_connection(("localhost", 5432), timeout=1):
            return f"postgresql+psycopg://{getpass.getuser()}@localhost:5432/postgres"
    except OSError:
        return None

PG = _local_pg()
if PG and adbc_fetch.driver_installed("postgresql"):
    from adbc_driver_postgresql import dbapi as pg
    admin = pg.connect(adbc_fetch.adbc_uri(__import__("sqlalchemy").engine.make_url(PG)), autocommit=True)
    with admin.cursor() as cur:
        cur.execute("drop table if exists adbc_notebook_demo")
        cur.execute("create table adbc_notebook_demo (id bigint, name text)")
        cur.execute("insert into adbc_notebook_demo values (1, 'a'), (2, 'b'), (3, 'c')")
    try:
        spec = SqlPartitionSpec(sql="select id, name from adbc_notebook_demo where id >= %(id_1)s order by id", params={"id_1": 2})
        pg_frame = SqlPartitionExecutor.fetch_partition(
            config=WorkerSqlConfig(connection_url=PG, query_only=True), gate_key="pg", max_concurrent_fetches=1,
            partition=spec, meta_dtypes={"id": "Int64", "name": "string"}, use_arrow=True,
        )
        print(pg_frame)
    finally:
        with admin.cursor() as cur:
            cur.execute("drop table if exists adbc_notebook_demo")
        admin.close()
else:
    print("No local PostgreSQL (or driver) available; skipped.")

select * from t where a = $1 and name like '%x%' and b in ($2, $3) (1, 'p', 'q')
   id name
0   2    b
1   3    c
